# World Models 3: MDN-RNN, Controller, and In-Dream Training

## Case study: planning in learned latent dynamics

This notebook trains a recurrent dynamics model on VAE latents, then optimizes a compact policy inside the model. The MDN-RNN receives the current latent and action, maintains hidden state, and predicts a distribution over the next latent plus reward and termination.

For each latent dimension, a mixture-density head predicts weights $\pi_k$, means $\mu_k$, and scales $\sigma_k$: $p(z_{t+1})=\sum_k\pi_k\mathcal N(z_{t+1};\mu_k,\sigma_k^2I)$. Mixtures can represent multiple plausible futures. Training minimizes mixture negative log-likelihood with auxiliary reward regression and done classification.

A controller maps latent and recurrent features to steering, throttle, and brake. CMA-ES searches controller parameters using ranked imagined returns. Full covariance adaptation can be expensive, so this notebook uses a lightweight diagonal evolution-strategy variant inspired by CMA-ES, not the canonical algorithm. Dream optimization is fast but vulnerable to model bias: an optimizer may exploit errors in the learned simulator. Imagined return must be checked in the real environment.

## 1. Load latent rollouts

We load the arrays produced by notebook 02. If they are missing, synthetic latent trajectories preserve the expected schema and let the complete training loop run. These artificial dynamics are not CarRacing physics.

**What this cell does:** loads or creates latent states, actions, rewards, terminal flags, and episode IDs.

In [ ]:
import random,math
from pathlib import Path
import numpy as np,torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
SEED=41;random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu');ROOT=Path('world-model-data')
ZDIM=32;ADIM=3;HID=96;K=5;T=16;LAT=ROOT/'latent_rollouts.npz'
def synth(n=100,horizon=40):
 z=[];a=[];r=[];d=[];e=[]
 for ep in range(n):
  x=np.random.randn(ZDIM).astype(np.float32)*.15;pos=np.random.uniform(-.5,.5)
  for t in range(horizon):
   u=np.array([np.random.uniform(-1,1),np.random.rand(),np.random.rand()*.1],np.float32);z.append(x.copy());a.append(u)
   pos=np.clip(pos+.04*u[0]+np.random.randn()*.01,-1.3,1.3);x=.92*x+np.random.randn(ZDIM).astype(np.float32)*.06;x[0]=pos
   r.append(.1-.4*(abs(pos)>1));done=t==horizon-1 or abs(pos)>1;d.append(float(done));e.append(ep)
   if done:break
 return tuple(map(np.asarray,(z,a,r,d,e)))
if LAT.exists():
 q=np.load(LAT);Z,A,R,D,EP=(q[k] for k in ('latents','actions','rewards','dones','episode_ids'));print('loaded latent rollouts',Z.shape)
else:Z,A,R,D,EP=synth();print('using synthetic latent trajectories',Z.shape)
Z=np.asarray(Z,np.float32);A=np.asarray(A,np.float32);R=np.asarray(R,np.float32);D=np.asarray(D,np.float32);EP=np.asarray(EP)
episodes=[np.flatnonzero(EP==e) for e in np.unique(EP) if np.sum(EP==e)>=T+1]


## 2. Episode-safe sequence batches

Training windows must not cross an episode boundary. We sample a sequence of latent/action inputs and next-latent, reward, and done targets within one episode.

**What this cell does:** builds a small sequence sampler and prints the tensor shapes.

In [ ]:
def batch(n=32):
 xs=[];zs=[];rs=[];ds=[]
 for _ in range(n):
  ep=random.choice(episodes); start=random.randrange(max(1,len(ep)-T)); ix=ep[start:min(start+T+1,len(ep))]
  if len(ix)<2:ix=ep[-2:]
  xs.append(np.concatenate([Z[ix[:-1]],A[ix[:-1]]],-1));zs.append(Z[ix[1:]]);rs.append(R[ix[1:]]);ds.append(D[ix[1:]])
 return tuple(torch.tensor(np.stack(v),dtype=torch.float32,device=device) for v in (xs,zs,rs,ds))
x,z,r,d=batch(8);print(x.shape,z.shape,r.shape,d.shape)


## 3. MDN-RNN architecture and likelihood

A GRU summarizes recent history. Its head predicts mixture logits, means, and log-scales. `logsumexp` computes the Gaussian-mixture likelihood stably; log-scales are clamped to avoid numerical extremes.

**What this cell does:** defines the recurrent model and the mixture negative log-likelihood.

In [ ]:
class MDNRNN(nn.Module):
 def __init__(self):
  super().__init__();self.gru=nn.GRU(ZDIM+ADIM,HID,batch_first=True);self.pi=nn.Linear(HID,K);self.mu=nn.Linear(HID,K*ZDIM);self.ls=nn.Linear(HID,K*ZDIM);self.reward=nn.Linear(HID,1);self.done=nn.Linear(HID,1)
 def forward(self,x,h=None):
  o,h=self.gru(x,h);b,t,_=o.shape
  return self.pi(o),self.mu(o).view(b,t,K,ZDIM),self.ls(o).view(b,t,K,ZDIM).clamp(-5,2),self.reward(o).squeeze(-1),self.done(o).squeeze(-1),h
def mdn_loss(logits,mu,ls,target):
 lp=-.5*(((target.unsqueeze(-2)-mu)/ls.exp())**2+2*ls+math.log(2*math.pi)).sum(-1)
 return -torch.logsumexp(F.log_softmax(logits,-1)+lp,-1).mean()
rnn=MDNRNN().to(device);out=rnn(x);print('mixture mean tensor:',out[1].shape)


## 4. Train the stochastic dynamics model

The objective combines latent mixture NLL, reward MSE, and done BCE. The terms have different scales, so coefficients balance them. We use gradient clipping and a bounded update count. Loss curves are useful diagnostics, not proof of accurate long-horizon simulation.

**What this cell does:** trains on sampled episode-safe windows and plots each loss component.

In [ ]:
opt=torch.optim.Adam(rnn.parameters(),lr=1e-3);UPDATES=300;logs=[]
for step in range(1,UPDATES+1):
 x,z,r,d=batch(32);pi,mu,ls,rp,dp,h=rnn(x);nll=mdn_loss(pi,mu,ls,z);rl=F.mse_loss(rp,r);dl=F.binary_cross_entropy_with_logits(dp,d);loss=nll+.5*rl+.25*dl
 opt.zero_grad(set_to_none=True);loss.backward();nn.utils.clip_grad_norm_(rnn.parameters(),1.);opt.step()
 if step%50==0:logs.append([step,nll.item(),rl.item(),dl.item()]);print(step,'MDN NLL',round(nll.item(),3),'reward MSE',round(rl.item(),3),'done BCE',round(dl.item(),3))
logs=np.array(logs);[plt.plot(logs[:,0],logs[:,i],label=l) for i,l in [(1,'latent NLL'),(2,'reward MSE'),(3,'done BCE')]];plt.xlabel('update');plt.grid();plt.legend();plt.show()


## 5. Controller and imagined rollout

The controller maps latent features plus eight recurrent-state features to continuous actions. The MDN samples a mixture component and next latent; its auxiliary heads estimate reward and termination. The rollout uses only the learned model after its initial latent state.

**What this cell does:** defines the policy parameterization and a discounted dream-return evaluator.

In [ ]:
OBS=ZDIM+8;NP=(OBS+1)*ADIM

def action(theta,z,h):
 w=theta[:OBS*ADIM].reshape(OBS,ADIM);b=theta[OBS*ADIM:].reshape(1,ADIM);f=torch.cat([z,h[:,:8]],-1);v=f@w+b
 return torch.stack([v[:,0].tanh(),v[:,1].sigmoid(),v[:,2].sigmoid()],-1)
@torch.no_grad()
def dream(theta,horizon=24):
 rnn.eval();total=[]
 for _ in range(2):
  i=random.randrange(len(Z));z=torch.tensor(Z[i:i+1],device=device);h=torch.zeros(1,1,HID,device=device);score=0.;discount=1.
  for t in range(horizon):
   u=action(theta,z,h[:,0,:]);pi,mu,ls,rp,dp,h=rnn(torch.cat([z,u],-1).unsqueeze(1),h)
   k=torch.distributions.Categorical(logits=pi[:,0]).sample();row=torch.arange(1,device=device);z=mu[:,0][row,k]+ls[:,0][row,k].exp()*torch.randn_like(mu[:,0][row,k])
   score+=discount*rp.item();discount*=.99
   if dp.sigmoid().item()>.7:break
  total.append(score)
 rnn.train();return float(np.mean(total))
initial=np.random.randn(NP).astype(np.float32)*.05
print('controller parameters:',NP,'initial dream return:',dream(torch.tensor(initial,device=device)))


## 6. CMA-ES-inspired controller search

CMA-ES adapts a search distribution based on ranked candidate fitness. The canonical method includes covariance and evolution-path adaptation; here we use a diagonal covariance evolution-strategy variant to keep computation small. Elite candidates shift the mean and adjust per-parameter scales. This is explicitly an educational approximation, not a canonical CMA-ES implementation.

**What this cell does:** searches controller parameters for a few generations in the learned dream and plots the best imagined score.

In [ ]:
def search(generations=7,population=10):
 mean=initial.copy();sigma=np.full_like(mean,.12);best=mean.copy();bestfit=-1e9;curve=[];elite=max(2,population//4)
 for g in range(generations):
  cand=mean+np.random.randn(population,len(mean)).astype(np.float32)*sigma
  fit=np.array([dream(torch.tensor(v,device=device)) for v in cand]);order=np.argsort(fit)[::-1];top=cand[order[:elite]]
  w=np.exp(np.linspace(0,-2,elite));w/=w.sum();new=(top*w[:,None]).sum(0);spread=np.sqrt((w[:,None]*(top-new)**2).sum(0)+1e-6)
  mean=new;sigma=np.clip(.75*sigma+.25*spread,.015,.35)
  if fit[order[0]]>bestfit:bestfit=fit[order[0]];best=cand[order[0]].copy()
  curve.append(bestfit);print(f'generation {g+1}/{generations}: best dream return {bestfit:.3f}')
 return best,np.array(curve)
best,curve=search();plt.plot(curve,marker='o');plt.xlabel('generation');plt.ylabel('best imagined return');plt.grid();plt.show()


## 7. In-dream training and model bias

A higher dream score means the controller found actions that score well according to the learned model. It does not establish better Box2D driving. Model errors compound over long rollouts, and optimization can exploit regions unsupported by training data. A sound workflow alternates between imagined optimization, real-environment evaluation, collecting policy rollouts, and retraining the world model.

**What this cell does:** compares initial and optimized controllers using the same model-based evaluator and states the evaluation limitation.

In [ ]:
start=dream(torch.tensor(initial,device=device),horizon=30);end=dream(torch.tensor(best,device=device),horizon=30)
print(f'initial imagined return: {start:.3f}');print(f'optimized imagined return: {end:.3f}')
print('These are model-based estimates, not real-environment scores.')


## Key takeaways

- An MDN-RNN predicts stochastic action-conditioned latent transitions, plus reward and termination.
- The VAE latent summarizes the current image; recurrent hidden state summarizes recent history.
- Sequence windows must respect episode boundaries.
- Search inside a learned model is fast, but errors compound and can be exploited.
- This notebook uses a lightweight diagonal evolution strategy inspired by CMA-ES, not full canonical CMA-ES.
- Evaluate the resulting policy in the real environment, compare baselines, collect new rollouts, and retrain.